# Lab Day 2 — Đào Gia Bảo · 2A202602793

Notebook triển khai từ khung của repo. Chọn cấu hình bằng macro-F1 val; test chỉ chạy sau khi chốt cấu hình và đủ ba seed. Báo cáo được sinh bằng tiếng Việt từ log thật.

Thứ tự: tải dữ liệu → EDA/kiểm tra → backbone → ablation → suy luận/độ trễ → chốt cấu hình → ba seed → test → workbook/báo cáo.


## 0. Cài đặt môi trường

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "eval.py").is_file()),
                Path("K4-Track4-Day2-Deeplearning-Advance").resolve())
if not (REPO_DIR / "eval.py").is_file():
    subprocess.run(["git", "clone", "https://github.com/kevindao94work/K4-Track4-Day2-Deeplearning-Advance.git", str(REPO_DIR)], check=True)
CODE_DIR = REPO_DIR / "submissions/2A202602793_dao_gia_bao/code"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(CODE_DIR / "requirements.txt")], check=True)
# Lần tái lập mới dùng thư mục riêng, giữ sản phẩm và lịch sử của bài nộp.
os.environ["LAB_OUTPUT_DIR"] = str(REPO_DIR / "reproduction")
os.environ["LAB_DATA_DIR"] = str(REPO_DIR / "data")
sys.path.insert(0, str(CODE_DIR))
sys.path.insert(0, str(REPO_DIR))
import torch, timm
print("torch", torch.__version__, "timm", timm.__version__)
print("CUDA", torch.cuda.is_available(), "MPS", torch.backends.mps.is_available())


### Tải dữ liệu nguồn chính thức
Trình tải giữ nguyên CSV fold 0 và kiểm tra MD5 `b7b30f96d466fba86016aa5a26606e0f` trước giải nén. ZIP chứa ảnh ở cấp gốc; trình tải đặt chúng vào `data/images`.


In [ ]:
from download_data import download
download(os.environ["LAB_DATA_DIR"])


In [ ]:
IMAGES_DIR = str(REPO_DIR / "data/images")
LABELS_DIR = str(REPO_DIR / "data/labels")


## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
from dataset import load_split, check_split
train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
split_info = check_split(train_df, val_df, test_df, IMAGES_DIR)
print("Thứ tự lớp:", split_info["class_names"])
print("Sai khác nhãn nguồn:", split_info["annotation_conflicts"])


In [ ]:
from validate_pipeline import main as validate_pipeline
validate_pipeline()
from IPython.display import display, Image
OUT_DIR = Path(os.environ["LAB_OUTPUT_DIR"])
for name in ["class_distribution.png", "class_samples.png", "augmentation_samples.png", "mixed_samples.png", "one_batch_overfit.png"]:
    display(Image(filename=str(OUT_DIR / "evidence" / name)))
print((OUT_DIR / "evidence/stage5.json").read_text())


## Bước 1 — So sánh năm backbone
Cùng công thức nền T00: 10 epoch, batch 32, 128×128, FP32, seed 0, fold 0. Các tag tiền huấn luyện được lưu trong log. Chỉ macro-F1 validation quyết định backbone đi tiếp; thời gian đo sơ bộ chưa thay cho benchmark chính thức.

In [ ]:
from workflow import backbones
backbone_choice = backbones()
display(__import__("pandas").read_csv(OUT_DIR / "tables/Backbones.csv"))
print("Backbone được chọn:", backbone_choice["source_exp_id"])

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# TODO: các trục A-G của GUIDE.md mục 3 (khởi tạo, augmentation, loss, sampler, LR/optimizer, EMA, ...).
# TODO: thử ít nhất một kết hợp các yếu tố tốt; so Δ với nhiễu (std).

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# TODO: TTA lật, multi-crop/scale, dò độ phân giải, gộp xác suất vs logit, ensemble, EMA/soup,
#       temperature scaling (T khớp trên VAL), gộp BN/FP16  (starter/inference.py)
# TODO: độ trễ p50/p95/p99 bằng starter/benchmark.py: warmup, synchronize, >= 50 lần

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# TODO: chạy chung kết và mốc, mỗi seed, ví dụ:
#   for seed in (0, 1, 2):
#       run(Config(exp_id="F01", ..., seed=seed, save_test_predictions=True))
#       run(Config(exp_id="T00", seed=seed, save_test_predictions=True))
# Với suy luận (TTA/ensemble/temperature scaling), tự ghi predictions/F01_seed<k>_test.csv
# bằng eval.save_predictions(...) từ xác suất cuối cùng; thêm F01uncal_* nếu muốn chấm I4(a).

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# TODO: ghi results.xlsx bằng pandas.ExcelWriter(engine="openpyxl") với các sheet:
#   Backbones, Training, Inference, Final, PerClass, Latency, Summary
# TODO: kiểm tra mỗi exp_id có ảnh trong curves/ và số trong xlsx khớp kết quả của eval.py